# Avaliação do Modelo de Risco de Crédito

Este notebook lê os artefatos produzidos por `Model/train.py` e apresenta seleção do algoritmo, métricas finais, matriz de confusão, impacto do threshold e explicabilidade.

Os helpers de leitura, resumo e visualização ficam em `DataPipeline/pipeline_functions.py`; o notebook permanece focado na apresentação e interpretação dos resultados.


### Bloco 1 — Preparação do ambiente
**O que este bloco faz:** localiza a raiz do projeto, importa bibliotecas e habilita a camada de storage para ler os artefatos do treinamento.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name in {"DataPipeline", "Model"}:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from DataPipeline import pipeline_functions as pf


### Bloco 2 — Metodologia do treinamento
**O que este bloco faz:** não executa código. Resume a sequência usada para manter o holdout fora da seleção do modelo.

1. Split estratificado 80/20.
2. CV-AUC para Logistic Regression, Random Forest e Gradient Boosting.
3. GridSearchCV somente no vencedor.
4. Fit final no conjunto de treino completo.
5. Avaliação única no holdout.
6. Geração de relatórios de threshold e explicabilidade.

### Bloco 3 — Carregamento das métricas
**O que este bloco faz:** lê `Model/metrics.json`, que contém algoritmo vencedor, resultado da seleção, hiperparâmetros e métricas finais do holdout.

In [ ]:
metrics = pf.load_metrics()
metrics


### Bloco 4 — Comparação dos algoritmos
**O que este bloco faz:** lê a CV-AUC dos três candidatos, mostra a tabela e cria um gráfico para visualizar qual algoritmo obteve a maior média na validação cruzada.

In [ ]:
comparison = pf.load_report("model_comparison")
display(comparison)
pf.plot_model_comparison(comparison)


### Bloco 5 — Métricas finais do vencedor
**O que este bloco faz:** extrai do JSON AUC-ROC, KS, Average Precision, recall, precision, F1, acurácia e threshold usados na avaliação final.

In [ ]:
summary = pf.model_metrics_summary(metrics)
display(summary.to_frame("valor"))


### Bloco 6 — Curva ROC
**O que este bloco faz:** lê os pontos da curva ROC calculados no holdout e plota TPR contra FPR, incluindo a linha de referência de um classificador aleatório.

In [ ]:
roc = pf.load_report("roc_curve")
pf.plot_roc_curve(roc, metrics["holdout"]["auc_roc"])


### Bloco 7 — Matriz de confusão
**O que este bloco faz:** reconstrói a matriz de confusão usando os valores salvos nas métricas e exibe VN, FP, FN e VP no threshold principal.

In [ ]:
matrix = pf.confusion_matrix_from_metrics(metrics)
pf.plot_confusion_matrix(matrix)


### Bloco 8 — Trade-off de threshold
**O que este bloco faz:** lê as políticas de threshold 0.30, 0.50 e 0.70 e mostra como elas alteram taxa de aprovação, recall, precision e falsos negativos.

In [ ]:
thresholds = pf.load_report("threshold_analysis")
display(thresholds)
pf.plot_threshold_tradeoff(thresholds)


### Bloco 9 — Importância nativa
**O que este bloco faz:** quando o modelo possui `feature_importances_` ou coeficientes, lê as 20 features de maior magnitude e exibe tabela e gráfico.

In [ ]:
if pf.report_exists("feature_importance"):
    native = pf.load_report("feature_importance").head(20)
    display(native)
    pf.plot_ranked_report(native, "abs_value", "Importância nativa — top 20")
else:
    print("Importância nativa não disponível.")


### Bloco 10 — Permutation Importance
**O que este bloco faz:** lê a queda média de AUC causada pelo embaralhamento de cada feature original. Quanto maior a queda, maior a dependência do modelo daquela variável.

In [ ]:
if pf.report_exists("permutation_importance"):
    permutation = pf.load_report("permutation_importance").head(20)
    display(permutation)
    pf.plot_ranked_report(permutation, "importance_mean", "Permutation importance — top 20")
else:
    print("Permutation importance não disponível.")


### Bloco 11 — SHAP global
**O que este bloco faz:** lê a média do valor absoluto dos SHAP values para mostrar quais features mais contribuíram para as previsões na amostra analisada.

In [ ]:
if pf.report_exists("shap_importance"):
    shap_importance = pf.load_report("shap_importance").head(20)
    display(shap_importance)
    pf.plot_ranked_report(shap_importance, "mean_abs_shap", "SHAP global — top 20")
elif pf.report_exists("shap_status"):
    print(pf.load_json_report("shap_status"))
else:
    print("SHAP não executado.")


### Bloco 12 — Interpretação das métricas
**O que este bloco faz:** não executa código. Resume como ler os principais resultados.

- **AUC-ROC:** capacidade de ordenar clientes de menor para maior risco.
- **KS:** separação entre as distribuições de adimplentes e inadimplentes.
- **Recall:** proporção dos inadimplentes reais capturados pelo threshold.
- **Precision:** proporção dos clientes marcados como risco que realmente são inadimplentes.
- **FN:** inadimplente classificado abaixo do threshold.
- **FP:** adimplente classificado acima do threshold.
- **Threshold:** regra que converte a probabilidade produzida pelo modelo em decisão operacional.